# B1 · XGBoost with early stopping on house prices

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 55 minutes · **core path ★**

## Objective

Train an `XGBRegressor` on California with **early stopping** on a validation set, compare three learning rates (0.3, 0.1, 0.03) for chosen rounds, RMSE and time, and place the result against linear regression, tree and random forest (previous modules). Check that halving the learning rate requires more rounds.

## Dataset

`ds.california()` split into training (60%), **validation** (20%, used only for early stopping) and test (20%, touched once): `X_tr, X_val, X_te` and `y_tr, y_val, y_te`. RMSE with `squared=False`.

## Educational purpose

Learn the standard workflow: low learning rate, many rounds allowed, `early_stopping_rounds`, best round read from `best_iteration`, test set only at the end. And understand the `learning_rate` ↔ number of rounds trade-off.

## Task

1. For `eta` in `[0.3, 0.1, 0.03]` train `XGBRegressor(n_estimators=4000, learning_rate=eta, max_depth=5, subsample=0.8, colsample_bytree=0.8, tree_method="hist", early_stopping_rounds=50, eval_metric="rmse", n_jobs=-1, random_state=0)` with `eval_set=[(X_val, y_val)]`. Save `table` (index `eta`, columns `best_round` (`best_iteration + 1`), `val_rmse` (`best_score`), `test_rmse` and `time_s`).
2. Save `eta_round_product` (`pd.Series`: `eta * best_round` for each learning rate) and `round_ratio_003_over_03` (rounds at 0.03 divided by rounds at 0.3).
3. Save `reference_rmse` (dictionary on the test set): `linear` (`LinearRegression`), `tree` (`DecisionTreeRegressor(min_samples_leaf=20)`), `forest` (`RandomForestRegressor(200, max_features=0.33, random_state=0, n_jobs=-1)`) and `xgb` (the best model of the three by `val_rmse`).
4. Save `val_test_gap` (`test_rmse − val_rmse` for the best model) and comment on it: it is small because the validation set was used only to stop.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
print(X_tr.shape, X_val.shape, X_te.shape)

## Your work

In [ ]:
rows = {}
models = {}
for eta in [0.3, 0.1, 0.03]:
    ...
table = pd.DataFrame(rows).T
table.round(4)

In [ ]:
eta_round_product = None
round_ratio_003_over_03 = None

In [ ]:
reference_rmse = {}
val_test_gap = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`m = xgb.XGBRegressor(...); m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)`; `m.best_iteration`, `m.best_score`; test RMSE: `mean_squared_error(y_te, m.predict(X_te), squared=False)` (with `early_stopping_rounds` the prediction already uses the best round).'):
    check(list(table.columns) == ["best_round", "val_rmse", "test_rmse", "time_s"] and list(table.index) == [0.3, 0.1, 0.03], "table with three learning rates and four columns", "best_round, val_rmse, test_rmse, time_s")
    check(table["best_round"].is_monotonic_increasing and table.loc[0.03, "best_round"] > 5 * table.loc[0.3, "best_round"], "lower learning rate = more rounds: at 0.03 over 5 times those at 0.3", "best_iteration + 1")
    check(table["test_rmse"].max() < 0.5 and table["test_rmse"].min() > 0.4 and table.loc[0.03, "test_rmse"] <= table.loc[0.3, "test_rmse"] + 0.005, "test RMSE between 0.40 and 0.50; the low learning rate does not get worse", "XGBRegressor with early stopping")
    check(close(eta_round_product.to_numpy(), table.index.to_numpy() * table["best_round"].to_numpy()) and eta_round_product.max() / eta_round_product.min() < 6 and round_ratio_003_over_03 > 5, "the eta x round product varies by less than 6 times across the learning rates (the rounds scale almost like 1/eta); round ratio over 5", "eta * best_round")
    check(reference_rmse["xgb"] < reference_rmse["forest"] < reference_rmse["tree"] < reference_rmse["linear"], "order: xgb < forest < tree < linear", "compare the test RMSEs")
    check(0 <= val_test_gap < 0.03, "the gap between test and validation RMSE is small (less than 0.03)", "the validation set only decided the number of rounds")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_xgboost_with_early_stopping_on_california.ipynb`. Open it after you pass the check, or after you have really tried.